In [5]:
:dep itertools = {version = "*"}
:dep serde = {version = "*", features = ["derive"]}

:dep polars = {version = "*" }
:dep polars-arrow = {version = "*" }

:dep pyo3-polars = { version = "0.20", features = ["derive"] }



In [33]:
// Version 35 — calc_ema with PlName variable for Series name
// Architecture: Copy (i64), single-state EMA, nulls from NaNs via from_vec_validity

// :dep polars = "0.39"
// :dep serde = { version = "1.0", features = ["derive"] }

// For plugin usage, uncomment:
// use pyo3_polars::derive::polars_expr;

use serde::Deserialize;
use polars::prelude::*;


#[derive(Deserialize)]
pub struct EmaKwargs {
    period: i64,
}

// #[polars_expr(output_type = Float64)]
fn calc_ema(input: &[Series], kwargs: EmaKwargs) -> PolarsResult<Series> {
    let series = input[0].cast(&DataType::Float64)?;
    let ca = series.f64().unwrap();
    let len = ca.len();
    let name ="ema";

    let period = kwargs.period;
    if period <= 0 {
        return Err(PolarsError::ComputeError("EMA period must be > 0".into()));
    }

    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema: f64 = f64::NAN;
    let mut count: i64 = 0;
    let mut out = vec![f64::NAN; len];

    for (i, opt_val) in ca.into_iter().enumerate() {
        let val = opt_val.unwrap_or(f64::NAN);

        if val.is_nan() {
            ema = f64::NAN;
            count = 0;
        } else if ema.is_nan() {
            ema = val;
            count = 1;
        } else {
            ema = alpha * val + (1.0 - alpha) * ema;
            count += 1;
        }

        if count >= period {
            out[i] = ema;
        }
    }

    // Convert NaNs in `out` to nulls using a validity bitmap
    let validity = out.iter().map(|v| !v.is_nan()).collect::<Vec<bool>>();
    let series = Float64Chunked::from_vec_validity(name, out, Some(validity.into())).into_series();

    // If you want to keep NaNs instead of nulls, use this instead:
    // let series = Float64Chunked::from_vec(name.as_ref(), out).into_series();

    Ok(series)
}

fn test_ema() -> PolarsResult<()> {
    let values = &[1.0, 2.0, 3.0, f64::NAN, 4.0, 8.0, 10.0];
    let input = Series::new("values", values);
    let kwargs = EmaKwargs { period: 3 };

    let mut ema = calc_ema(&[input.clone()], kwargs)?;
    ema.rename("ema");

    println!("Input Series:\n{}", input);
    println!("EMA Series:\n{}", ema);

    Ok(())
}

In [34]:
test_ema()

Input Series:
shape: (7,)
Series: 'values' [f64]
[
	1.0
	2.0
	3.0
	NaN
	4.0
	8.0
	10.0
]
EMA Series:
shape: (7,)
Series: 'ema' [f64]
[
	null
	null
	2.25
	null
	null
	null
	8.0
]


Ok(())

In [6]:
// Version 36 — calc_ema using PlSmallStr with from_vec_validity
// Architecture: Copy (i64), single-state EMA, nulls from NaNs via from_vec_validity

// :dep polars = "0.39"
// :dep serde = { version = "1.0", features = ["derive"] }

// For plugin usage, uncomment:
// use pyo3_polars::derive::polars_expr;

use polars::prelude::*;
use serde::Deserialize;

#[derive(Deserialize)]
pub struct EmaKwargs {
    period: i64,
}

// #[polars_expr(output_type = Float64)]
fn calc_ema(input: &[Series], kwargs: EmaKwargs) -> PolarsResult<Series> {
    let series = input[0].cast(&DataType::Float64)?;
    let ca = series.f64().unwrap();
    let len = ca.len();
    let name = "ema";

    let period = kwargs.period;
    if period <= 0 {
        return Err(PolarsError::ComputeError("EMA period must be > 0".into()));
    }

    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema: f64 = f64::NAN;
    let mut count: i64 = 0;
    let mut out = vec![f64::NAN; len];

    for (i, opt_val) in ca.into_iter().enumerate() {
        let val = opt_val.unwrap_or(f64::NAN);

        if val.is_nan() {
            ema = f64::NAN;
            count = 0;
        } else if ema.is_nan() {
            ema = val;
            count = 1;
        } else {
            ema = alpha * val + (1.0 - alpha) * ema;
            count += 1;
        }

        if count >= period {
            out[i] = ema;
        }
    }

    // Convert NaNs in `out` to nulls using a validity bitmap
    let validity = out.iter().map(|v| !v.is_nan()).collect::<Vec<bool>>();
    let series = Float64Chunked::from_vec_validity(name.into(), out, Some(validity.into())).into_series();

    // If you want to keep NaNs instead of nulls, use this instead:
    // let series = Float64Chunked::from_vec(name.as_ref(), out).into_series();

    Ok(series)
}

fn test_ema() -> PolarsResult<()> {
    let values = &[1.0, 2.0, 3.0, f64::NAN, 4.0, 8.0, 10.0];
    let input = Series::new("values".into(), values);
    let kwargs = EmaKwargs { period: 3 };

    let mut ema = calc_ema(&[input.clone()], kwargs)?;
    ema.rename("ema".into());

    println!("Input Series:\n{}", input);
    println!("EMA Series:\n{}", ema);

    Ok(())
}

In [4]:
// Version 47 — renamed input_series to series in compute_ema
// Architecture: Copy (i64), single-state EMA, nulls from NaNs via from_vec_validity

// :dep polars = "0.39"
// :dep serde = { version = "1.0", features = ["derive"] }

// For plugin usage, uncomment:
// use pyo3_polars::derive::polars_expr;

use polars::prelude::*;
use serde::Deserialize;

#[derive(Deserialize)]
pub struct EmaKwargs {
    period: i64,
}

// #[polars_expr(output_type = Float64)]
fn ema_expr(inputs: &[Series], kwargs: EmaKwargs) -> PolarsResult<Series> {
    compute_ema(&inputs[0], kwargs.period)
}

fn compute_ema(series: &Series, period: i64) -> PolarsResult<Series> {
    let series = series.cast(&DataType::Float64)?;
    let ca = series.f64().unwrap();
    let len = ca.len();
    let name = "ema";

    if period <= 0 {
        return Err(PolarsError::ComputeError("EMA period must be > 0".into()));
    }

    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema: f64 = f64::NAN;
    let mut count: i64 = 0;
    let mut out = vec![f64::NAN; len];

    for (i, opt_val) in ca.into_iter().enumerate() {
        let val = opt_val.unwrap_or(f64::NAN);

        if val.is_nan() {
            ema = f64::NAN;
            count = 0;
        } else if ema.is_nan() {
            ema = val;
            count = 1;
        } else {
            ema = alpha * val + (1.0 - alpha) * ema;
            count += 1;
        }

        if count >= period {
            out[i] = ema;
        }
    }

    // Convert NaNs in `out` to nulls using a validity bitmap
    let validity = out.iter().map(|v| !v.is_nan()).collect::<Vec<bool>>();
    let output = Float64Chunked::from_vec_validity(name.into(), out, Some(validity.into())).into_series();

    Ok(output)
}

fn test_ema() -> PolarsResult<()> {
    let values = &[1.0, 2.0, 3.0, f64::NAN, 4.0, 8.0, 10.0];
    let series = Series::new("values".into(), values);
    let inputs = &[series.clone()];
    let kwargs = EmaKwargs { period: 3 };

    let mut ema = ema_expr(inputs, kwargs)?;
    ema.rename("ema".into());

    println!("Input Series:\n{}", series);
    println!("EMA Series:\n{}", ema);

    Ok(())
}

In [5]:
test_ema()

Input Series:
shape: (7,)
Series: 'values' [f64]
[
	1.0
	2.0
	3.0
	NaN
	4.0
	8.0
	10.0
]
EMA Series:
shape: (7,)
Series: 'ema' [f64]
[
	null
	null
	2.25
	null
	null
	null
	8.0
]


Ok(())

In [6]:
// Version 48 — use `ema +=` in update step
// Architecture: Copy (i64), single-state EMA, nulls from NaNs via from_vec_validity

// :dep polars = "0.39"
// :dep serde = { version = "1.0", features = ["derive"] }

// For plugin usage, uncomment:
// use pyo3_polars::derive::polars_expr;

use polars::prelude::*;
use serde::Deserialize;

#[derive(Deserialize)]
pub struct EmaKwargs {
    period: i64,
}

// #[polars_expr(output_type = Float64)]
fn ema_expr(inputs: &[Series], kwargs: EmaKwargs) -> PolarsResult<Series> {
    compute_ema(&inputs[0], kwargs.period)
}

fn compute_ema(series: &Series, period: i64) -> PolarsResult<Series> {
    let series = series.cast(&DataType::Float64)?;
    let ca = series.f64().unwrap();
    let len = ca.len();
    let name = "ema";

    if period <= 0 {
        return Err(PolarsError::ComputeError("EMA period must be > 0".into()));
    }

    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema: f64 = f64::NAN;
    let mut count: i64 = 0;
    let mut out = vec![f64::NAN; len];

    for (i, opt_val) in ca.into_iter().enumerate() {
        let val = opt_val.unwrap_or(f64::NAN);

        if val.is_nan() {
            ema = f64::NAN;
            count = 0;
        } else if ema.is_nan() {
            ema = val;
            count = 1;
        } else {
            ema += alpha * (val - ema); // ← simplified using ema += form
            count += 1;
        }

        if count >= period {
            out[i] = ema;
        }
    }

    // Convert NaNs in `out` to nulls using a validity bitmap
    let validity = out.iter().map(|v| !v.is_nan()).collect::<Vec<bool>>();
    let output = Float64Chunked::from_vec_validity(name.into(), out, Some(validity.into())).into_series();

    Ok(output)
}

fn test_ema() -> PolarsResult<()> {
    let values = &[1.0, 2.0, 3.0, f64::NAN, 4.0, 8.0, 10.0];
    let series = Series::new("values".into(), values);
    let inputs = &[series.clone()];
    let kwargs = EmaKwargs { period: 3 };

    let mut ema = ema_expr(inputs, kwargs)?;
    ema.rename("ema".into());

    println!("Input Series:\n{}", series);
    println!("EMA Series:\n{}", ema);

    Ok(())
}

In [8]:
test_ema()

Input Series:
shape: (7,)
Series: 'values' [f64]
[
	1.0
	2.0
	3.0
	NaN
	4.0
	8.0
	10.0
]
EMA Series:
shape: (7,)
Series: 'ema' [f64]
[
	null
	null
	2.25
	null
	null
	null
	8.0
]


Ok(())

In [6]:
// Version 49 — renamed compute_ema to raw_ema
// Architecture: Copy (i64), single-state EMA, nulls from NaNs via from_vec_validity

// :dep polars = "0.39"
// :dep serde = { version = "1.0", features = ["derive"] }

// For plugin usage, uncomment:
use pyo3_polars::derive::polars_expr;

use polars::prelude::*;
use serde::Deserialize;

#[derive(Deserialize)]
pub struct EmaKwargs {
    period: i64,
}

#[polars_expr(output_type = Float64)]
fn ema_expr(inputs: &[Series], kwargs: EmaKwargs) -> PolarsResult<Series> {
    raw_ema(&inputs[0], kwargs.period)
}

// #[polars_expr(output_type = Float64)]
fn ema_expr2(inputs: &[Series], kwargs: EmaKwargs) -> PolarsResult<Series> {
    raw_ema(&inputs[0], kwargs.period)
}


fn raw_ema(series: &Series, period: i64) -> PolarsResult<Series> {
    let series = series.cast(&DataType::Float64)?;
    let ca = series.f64().unwrap();
    let len = ca.len();
    let name = "ema";

    if period <= 0 {
        return Err(PolarsError::ComputeError("EMA period must be > 0".into()));
    }

    let alpha = 2.0 / (period as f64 + 1.0);
    let mut ema: f64 = f64::NAN;
    let mut count: i64 = 0;
    let mut out = vec![f64::NAN; len];

    for (i, opt_val) in ca.into_iter().enumerate() {
        let val = opt_val.unwrap_or(f64::NAN);

        if val.is_nan() {
            ema = f64::NAN;
            count = 0;
        } else if ema.is_nan() {
            ema = val;
            count = 1;
        } else {
            ema += alpha * (val - ema);
            count += 1;
        }

        if count >= period {
            out[i] = ema;
        }
    }

    // Convert NaNs in `out` to nulls using a validity bitmap
    let validity = out.iter().map(|v| !v.is_nan()).collect::<Vec<bool>>();
    let output = Float64Chunked::from_vec_validity(name.into(), out, Some(validity.into())).into_series();

    Ok(output)
}

fn test_ema() -> PolarsResult<()> {
    let values = &[1.0, 2.0, 3.0, f64::NAN, 4.0, 8.0, 10.0];
    let series = Series::new("values".into(), values);
    let inputs = &[series.clone()];
    let kwargs = EmaKwargs { period: 3 };

    let mut ema = ema_expr2(inputs, kwargs)?;
    ema.rename("ema".into());

    println!("Input Series:\n{}", series);
    println!("EMA Series:\n{}", ema);

    Ok(())
}

Error: mismatched types

Error: mismatched types

Error: mismatched types

Error: mismatched types

In [10]:
test_ema()


Input Series:
shape: (7,)
Series: 'values' [f64]
[
	1.0
	2.0
	3.0
	NaN
	4.0
	8.0
	10.0
]
EMA Series:
shape: (7,)
Series: 'ema' [f64]
[
	null
	null
	2.25
	null
	null
	null
	8.0
]


Ok(())